In [ ]:
# Define coordinates as a list of [lon, lat]
coordinates_list = [
    # [lon, lat], # Add more coordinates as needed
    [91.2734,   29.7904], # 拉萨
    [91.9113,   29.1816], # 山南
    [92.6035,   28.9443], # 加查
    [94.2605,   29.7733], # 尼西村
    [94.3610,   29.6483], # 林芝
    [100.1120,  25.6540], # 苍山
    [100.7173,  22.1298], # 西双版纳
    [101.2663,  24.3681], # 景东
    [102.2691,  27.9628], # 凉山
    [104.6592,  29.2588], # 自贡
    [109.0416,  24.3102], # 柳州 云太多
    [115.8977,  33.0074], # 阜阳 云太多
    [116.3372,  31.3563], # 南岳
    [118.1656,  30.1375], # 黄山
    [77.6127,   22.1257],
    [108.6054,  29.1993],
    [88.9622,   30.9770],
    [102.1833,  44.3633],
    [32.4185,   24.8692],
    [17.5781,   -2.1907],
    [-41.7397,  -5.5464],
    [-85.8762,  34.0330],
    [-101.3378, 37.2592],
    [-107.2751, 51.4117],
    [23.5720,   -3.4912],
    [-62.2159,  -3.4653],
    [124.7115,  52.3352],
    [-118.1230, 34.2197],
    [150.3557,  -33.3042],
]

# Image Size in pixels
pixel_size = 1024

image_collection_id = "COPERNICUS/S2_HARMONIZED"
bands_to_export = [
    "B2", # Blue
    "B3", # Green
    "B4", # Red
    "B8", # NIR
    "B11",# SWIR1
    "B12" # SWIR2
]
start_date = "2015-01-01"
end_date = "2026-12-31"
use_csplus = True
csplus_thresh = 0.8
cloud_filter_percentage = 20
export_scale = 10
export_crs = "EPSG:4326"
max_pixels = 1e13

# Export destination. This notebook now uses Earth Engine batch exports directly.
drive_folder = "sentinel-2"
drive_file_prefix = ""

# Export angle cubes whose time slices align with the optical-band cubes.
# Sentinel-2 exposes scene-level mean solar angles in Earth Engine metadata.
# Solar altitude is computed as 90 - MEAN_SOLAR_ZENITH_ANGLE.
export_solar_angle_cubes = True
angle_bands_to_export = ["SOLAR_ALTITUDE", "SOLAR_AZIMUTH"]

# Solar angle slices are masked to valid reflectance pixels so each pixel keeps
# the solar angle of the image that actually contributed its observation.
# "all" means the angle pixel is valid only when all exported reflectance bands are valid.
# Use "any" if you want angle pixels wherever at least one exported band is valid.
solar_angle_mask_bands = bands_to_export
solar_angle_mask_mode = "all"


In [8]:
import ee
from ee.geometry import Geometry
from ee.imagecollection import ImageCollection
from ee.image import Image
from ee.filter import Filter
from ee.join import Join

try:
    ee.Initialize(project="ee-yangluhao990714")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-yangluhao990714")


In [9]:
def collection_name_for_file(collection_id: str) -> str:
    return collection_id.replace("/", "_")


def task_description(name: str) -> str:
    return name.replace(".", "_").replace("-", "_")[:100]


def add_csplus_mask(collection: ImageCollection, threshold: float) -> ImageCollection:
    cs = ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    join_filter = Filter.equals(leftField="system:index", rightField="system:index")
    join = Join.saveFirst(matchKey="csplus_mask")
    joined = join.apply(collection, cs, join_filter)

    def add_mask(img):
        img = ee.Image(img)
        cs_img = ee.Image(img.get("csplus_mask"))
        return img.updateMask(cs_img.select("cs").gt(threshold))

    return ee.ImageCollection(joined.map(add_mask))


def valid_observation_mask(img):
    masks = img.select(solar_angle_mask_bands).mask()
    if solar_angle_mask_mode == "any":
        return masks.reduce(ee.Reducer.max())
    if solar_angle_mask_mode == "all":
        return masks.reduce(ee.Reducer.min())
    raise ValueError("solar_angle_mask_mode must be 'all' or 'any'")


def solar_angle_image(img):
    img = ee.Image(img)
    valid_mask = valid_observation_mask(img)
    altitude = ee.Image.constant(90).subtract(
        ee.Image.constant(img.getNumber("MEAN_SOLAR_ZENITH_ANGLE"))
    ).rename("SOLAR_ALTITUDE")
    azimuth = ee.Image.constant(img.getNumber("MEAN_SOLAR_AZIMUTH_ANGLE")).rename(
        "SOLAR_AZIMUTH"
    )
    return (
        altitude.addBands(azimuth)
        .updateMask(valid_mask)
        .toFloat()
        .copyProperties(img, ["system:time_start", "system:index"])
        .set("system:index", img.get("system:index"))
    )


def export_image_to_drive(image, file_name: str, region):
    task = ee.batch.Export.image.toDrive(
        image=image.toFloat(),
        description=task_description(file_name),
        folder=drive_folder,
        fileNamePrefix=f"{drive_file_prefix}{file_name}",
        region=region,
        scale=export_scale,
        crs=export_crs,
        maxPixels=max_pixels,
        fileFormat="GeoTIFF",
    )
    task.start()
    print(f"Started Drive export: {file_name} -> Google Drive/{drive_folder}")
    return task


tasks = []

for coords in coordinates_list:
    lon, lat = coords[0], coords[1]

    # Calculate buffer size in meters to get desired pixel dimensions.
    buffer_size = (pixel_size * export_scale) / 2
    point = Geometry.Point([lon, lat])
    bbox = point.buffer(buffer_size).bounds()

    image_collection: ImageCollection = (
        ImageCollection(image_collection_id)
        .filterBounds(bbox)
        .filterDate(start_date, end_date)
    )

    if cloud_filter_percentage is not None:
        image_collection = image_collection.filter(
            Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_filter_percentage)
        )

    if use_csplus:
        image_collection = add_csplus_mask(image_collection, csplus_thresh)

    image_collection = image_collection.sort("system:time_start")

    collection_token = collection_name_for_file(image_collection_id)

    for band_name in bands_to_export:
        time_series_cube = image_collection.select(band_name).toBands()
        file_name = f"{collection_token}_{band_name}_lon{lon:.4f}_lat{lat:.4f}"
        try:
            tasks.append(export_image_to_drive(time_series_cube, file_name, bbox))
        except Exception as exc:
            print(f"Failed to start export: {file_name}: {exc}")

    if export_solar_angle_cubes:
        angle_collection = ee.ImageCollection(image_collection.map(solar_angle_image))
        for angle_band in angle_bands_to_export:
            angle_cube = angle_collection.select(angle_band).toBands()
            file_name = f"{collection_token}_{angle_band}_lon{lon:.4f}_lat{lat:.4f}"
            try:
                tasks.append(export_image_to_drive(angle_cube, file_name, bbox))
            except Exception as exc:
                print(f"Failed to start export: {file_name}: {exc}")

print(f"Started {len(tasks)} Earth Engine Drive export tasks.")


Started Drive export: COPERNICUS_S2_HARMONIZED_B2_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B3_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B4_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B8_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B11_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B12_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_SOLAR_ALTITUDE_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_SOLAR_AZIMUTH_lon91.2734_lat29.7904 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B2_lon91.9113_lat29.1816 -> Google Drive/sentinel-2
Started Drive export: COPERNICUS_S2_HARMONIZED_B3_lon91.9113_lat29.1816 -> Google Dr